In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/house-prices-advanced-regression-techniques/test.csv


In [2]:

import warnings
warnings.simplefilter(action='ignore', category=Warning)
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
import sys
np.set_printoptions(threshold=sys.maxsize)

In [3]:
train_df=pd.read_csv('../input/house-prices-advanced-regression-techniques/train.csv' ,index_col= 'Id')
test_df= pd.read_csv('../input/house-prices-advanced-regression-techniques/test.csv',index_col= 'Id')


# **check the Data ***

In [4]:

print(test_df.shape)
print(train_df.shape)

(1459, 79)
(1460, 80)


In [5]:
train_df.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
Id,,,,,,,,,,,,,,,,,,,,,
1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


# **Finding features with most number of missing values**

In [6]:

train_df.isnull().sum().nlargest(20)


PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrType         8
MasVnrArea         8
Electrical         1
MSSubClass         0
dtype: int64

# **Drop columns with high amount of null objects from train Data**

In [7]:
train_df=train_df.drop(['Alley','PoolQC','Fence','MiscFeature','FireplaceQu'], axis=1)

In [8]:
test_df.isnull().sum().nlargest(20)

PoolQC          1456
MiscFeature     1408
Alley           1352
Fence           1169
FireplaceQu      730
LotFrontage      227
GarageYrBlt       78
GarageFinish      78
GarageQual        78
GarageCond        78
GarageType        76
BsmtCond          45
BsmtQual          44
BsmtExposure      44
BsmtFinType1      42
BsmtFinType2      42
MasVnrType        16
MasVnrArea        15
MSZoning           4
Utilities          2
dtype: int64

**drop same columns from test data**

In [9]:
test_df=test_df.drop(['Alley','PoolQC','Fence','MiscFeature','FireplaceQu'], axis=1)

# **Finding most important features (based on correlation)**

In [10]:
top_corr_features = train_df.corr()['SalePrice'].sort_values(ascending=False)
top_corr_features

SalePrice        1.000000
OverallQual      0.790982
GrLivArea        0.708624
GarageCars       0.640409
GarageArea       0.623431
TotalBsmtSF      0.613581
1stFlrSF         0.605852
FullBath         0.560664
TotRmsAbvGrd     0.533723
YearBuilt        0.522897
YearRemodAdd     0.507101
GarageYrBlt      0.486362
MasVnrArea       0.477493
Fireplaces       0.466929
BsmtFinSF1       0.386420
LotFrontage      0.351799
WoodDeckSF       0.324413
2ndFlrSF         0.319334
OpenPorchSF      0.315856
HalfBath         0.284108
LotArea          0.263843
BsmtFullBath     0.227122
BsmtUnfSF        0.214479
BedroomAbvGr     0.168213
ScreenPorch      0.111447
PoolArea         0.092404
MoSold           0.046432
3SsnPorch        0.044584
BsmtFinSF2      -0.011378
BsmtHalfBath    -0.016844
MiscVal         -0.021190
LowQualFinSF    -0.025606
YrSold          -0.028923
OverallCond     -0.077856
MSSubClass      -0.084284
EnclosedPorch   -0.128578
KitchenAbvGr    -0.135907
Name: SalePrice, dtype: float64

In [11]:
Correlation_Matrix = train_df.corr()
Correlation_Matrix.style.background_gradient(cmap='coolwarm')

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,TotRmsAbvGrd,Fireplaces,GarageYrBlt,GarageCars,GarageArea,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SalePrice
MSSubClass,1.000000,-0.386347,-0.139781,0.032628,-0.059316,0.027850,0.040581,0.022936,-0.069836,-0.065649,-0.140759,-0.238518,-0.251758,0.307886,0.046474,0.074853,0.003491,-0.002333,0.131608,0.177354,-0.023438,0.281721,0.040380,-0.045569,0.085072,-0.040110,-0.098672,-0.012579,-0.006100,-0.012037,-0.043825,-0.026030,0.008283,-0.007683,-0.013585,-0.021407,-0.084284
LotFrontage,-0.386347,1.000000,0.426095,0.251646,-0.059213,0.123349,0.088866,0.193458,0.233633,0.049900,0.132644,0.392075,0.457181,0.080177,0.038469,0.402797,0.100949,-0.007234,0.198769,0.053532,0.263170,-0.006069,0.352096,0.266639,0.070250,0.285691,0.344997,0.088521,0.151972,0.010700,0.070029,0.041383,0.206167,0.003368,0.011200,0.007450,0.351799
LotArea,-0.139781,0.426095,1.000000,0.105806,-0.005636,0.014228,0.013788,0.104160,0.214103,0.111170,-0.002618,0.260833,0.299475,0.050986,0.004779,0.263116,0.158155,0.048046,0.126031,0.014259,0.119690,-0.017784,0.190015,0.271364,-0.024947,0.154871,0.180403,0.171698,0.084774,-0.018340,0.020423,0.043160,0.077672,0.038068,0.001205,-0.014261,0.263843
OverallQual,0.032628,0.251646,0.105806,1.000000,-0.091932,0.572323,0.550684,0.411876,0.239666,-0.059119,0.308159,0.537808,0.476224,0.295493,-0.030429,0.593007,0.111098,-0.040150,0.550600,0.273458,0.101676,-0.183882,0.427452,0.396765,0.547766,0.600671,0.562022,0.238923,0.308819,-0.113937,0.030371,0.064886,0.065166,-0.031406,0.070815,-0.027347,0.790982
OverallCond,-0.059316,-0.059213,-0.005636,-0.091932,1.000000,-0.375983,0.073741,-0.128101,-0.046231,0.040229,-0.136841,-0.171098,-0.144203,0.028942,0.025494,-0.079686,-0.054942,0.117821,-0.194149,-0.060769,0.012980,-0.087001,-0.057583,-0.023820,-0.324297,-0.185758,-0.151521,-0.003334,-0.032589,0.070356,0.025504,0.054811,-0.001985,0.068777,-0.003511,0.043950,-0.077856
YearBuilt,0.027850,0.123349,0.014228,0.572323,-0.375983,1.000000,0.592855,0.315707,0.249503,-0.049107,0.149040,0.391452,0.281986,0.010308,-0.183784,0.199010,0.187599,-0.038162,0.468271,0.242656,-0.070651,-0.174800,0.095589,0.147716,0.825667,0.537850,0.478954,0.224880,0.188686,-0.387268,0.031355,-0.050364,0.004950,-0.034383,0.012398,-0.013618,0.522897
YearRemodAdd,0.040581,0.088866,0.013788,0.550684,0.073741,0.592855,1.000000,0.179618,0.128451,-0.067759,0.181133,0.291066,0.240379,0.140024,-0.062419,0.287389,0.119470,-0.012337,0.439046,0.183331,-0.040581,-0.149598,0.191740,0.112581,0.642277,0.420622,0.371600,0.205726,0.226298,-0.193919,0.045286,-0.038740,0.005829,-0.010286,0.021490,0.035743,0.507101
MasVnrArea,0.022936,0.193458,0.104160,0.411876,-0.128101,0.315707,0.179618,1.000000,0.264736,-0.072319,0.114442,0.363936,0.344501,0.174561,-0.069071,0.390857,0.085310,0.026673,0.276833,0.201444,0.102821,-0.037610,0.280682,0.249070,0.252691,0.364204,0.373066,0.159718,0.125703,-0.110204,0.018796,0.061466,0.011723,-0.029815,-0.005965,-0.008201,0.477493
BsmtFinSF1,-0.069836,0.233633,0.214103,0.239666,-0.046231,0.249503,0.128451,0.264736,1.000000,-0.050117,-0.495251,0.522396,0.445863,-0.137079,-0.064503,0.208171,0.649212,0.067418,0.058543,0.004262,-0.107355,-0.081007,0.044316,0.260011,0.153484,0.224054,0.296970,0.204306,0.111761,-0.102303,0.026451,0.062021,0.140491,0.003571,-0.015727,0.014359,0.386420
BsmtFinSF2,-0.065649,0.049900,0.111170,-0.059119,0.040229,-0.049107,-0.067759,-0.072319,-0.050117,1.000000,-0.209294,0.104810,0.097117,-0.099260,0.014807,-0.009640,0.158678,0.070948,-0.076444,-0.032148,-0.015728,-0.040751,-0.035227,0.046921,-0.088011,-0.038264,-0.018227,0.067898,0.003093,0.036543,-0.029993,0.088871,0.041709,0.004940,-0.015211,0.031706,-0.011378


# **We still have columns with large null values but we do not remove them because they have high corr..like "LotFrontage"**

In [12]:
train_df.isnull().sum().nlargest(15)

LotFrontage     259
GarageType       81
GarageYrBlt      81
GarageFinish     81
GarageQual       81
GarageCond       81
BsmtExposure     38
BsmtFinType2     38
BsmtQual         37
BsmtCond         37
BsmtFinType1     37
MasVnrType        8
MasVnrArea        8
Electrical        1
MSSubClass        0
dtype: int64

In [13]:
low_corr_cols=['MoSold','3SsnPorch', 'BsmtFinSF2','BsmtHalfBath','MiscVal','LowQualFinSF','YrSold']

In [14]:
train_df=train_df.drop(low_corr_cols , axis=1)
test_df=test_df.drop(low_corr_cols , axis=1)

# **After Deleting low_corr and high_number_null columns , we have to categorize the features to Impute missing values**
# 

In [15]:
categorical_feature = ['MSZoning','Street', 'LotShape','LandContour',
 'LotConfig', 'LandSlope', 'RoofStyle','RoofMatl','MasVnrType', 'ExterCond', 'BsmtCond', 'Heating', 'Functional', 'GarageType', 'GarageYrBlt', 'GarageFinish',
 'GarageQual','GarageCond', 'PavedDrive']

ordinal_feature = ["BsmtQual" ,"Foundation" ,"BsmtExposure"  ,"BsmtFinType1" ,"BsmtFinType2", "HeatingQC" ,"CentralAir" ,"Electrical" ,"KitchenQual"]
numerical_feature = ["LotFrontage" , "LotArea" ,"BsmtFinSF1" ,"1stFlrSF" , "2ndFlrSF" , "FullBath",
                     "TotRmsAbvGrd","ScreenPorch", "PoolArea", "OverallQual","OverallCond" ,"YearRemodAdd"
                   ,"TotalBsmtSF" ,"GarageCars",'MasVnrArea']
all_features=categorical_feature+ordinal_feature+numerical_feature

In [16]:
def Impute_Data(df , type_df):
  if type_df=='numerical':
        imp_mean = SimpleImputer(missing_values=np.nan, strategy='mean')
        imp_mean.fit(df)
        df=imp_mean.transform(df)
        return df
  elif type_df in ['ordinal','categorical']:
        imp_mod = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
        imp_mod.fit(df)
        df=imp_mod.transform(df)
        return df
    

In [17]:
train_df=train_df[categorical_feature+ordinal_feature+numerical_feature+['SalePrice']]
test_df=test_df[categorical_feature+ordinal_feature+numerical_feature]

In [18]:
train_df[categorical_feature]=Impute_Data(train_df[categorical_feature],'categorical')
train_df[ordinal_feature]=Impute_Data(train_df[ordinal_feature],'ordinal')
train_df[numerical_feature]=Impute_Data(train_df[numerical_feature],'numerical')

In [19]:
train_df.head()

,MSZoning,Street,LotShape,LandContour,LotConfig,LandSlope,RoofStyle,RoofMatl,MasVnrType,ExterCond,...,TotRmsAbvGrd,ScreenPorch,PoolArea,OverallQual,OverallCond,YearRemodAdd,TotalBsmtSF,GarageCars,MasVnrArea,SalePrice
Id,,,,,,,,,,,,,,,,,,,,,
1,RL,Pave,Reg,Lvl,Inside,Gtl,Gable,CompShg,BrkFace,TA,...,8.0,0.0,0.0,7.0,5.0,2003.0,856.0,2.0,196.0,208500
2,RL,Pave,Reg,Lvl,FR2,Gtl,Gable,CompShg,None,TA,...,6.0,0.0,0.0,6.0,8.0,1976.0,1262.0,2.0,0.0,181500
3,RL,Pave,IR1,Lvl,Inside,Gtl,Gable,CompShg,BrkFace,TA,...,6.0,0.0,0.0,7.0,5.0,2002.0,920.0,2.0,162.0,223500
4,RL,Pave,IR1,Lvl,Corner,Gtl,Gable,CompShg,None,TA,...,7.0,0.0,0.0,7.0,5.0,1970.0,756.0,3.0,0.0,140000
5,RL,Pave,IR1,Lvl,FR2,Gtl,Gable,CompShg,BrkFace,TA,...,9.0,0.0,0.0,8.0,5.0,2000.0,1145.0,3.0,350.0,250000


In [20]:
test_df[categorical_feature]=Impute_Data(test_df[categorical_feature],'categorical')
test_df[ordinal_feature]=Impute_Data(test_df[ordinal_feature],'ordinal')
test_df[numerical_feature]=Impute_Data(test_df[numerical_feature],'numerical')

# **check again for missing value**

In [21]:
train_df.isnull().sum().nlargest(15)

MSZoning       0
Street         0
LotShape       0
LandContour    0
LotConfig      0
LandSlope      0
RoofStyle      0
RoofMatl       0
MasVnrType     0
ExterCond      0
BsmtCond       0
Heating        0
Functional     0
GarageType     0
GarageYrBlt    0
dtype: int64

In [22]:

test_df.isnull().sum().nlargest(15)


MSZoning       0
Street         0
LotShape       0
LandContour    0
LotConfig      0
LandSlope      0
RoofStyle      0
RoofMatl       0
MasVnrType     0
ExterCond      0
BsmtCond       0
Heating        0
Functional     0
GarageType     0
GarageYrBlt    0
dtype: int64

# **Now we should turn object data to numbers ! **

In [23]:
Onehot_Encoding = OneHotEncoder(sparse=False)
features_Onehot_encoded = pd.DataFrame(Onehot_Encoding.fit_transform(train_df[categorical_feature].astype(str) ))
features_Onehot_encoded.columns = Onehot_Encoding.get_feature_names(categorical_feature)
train_df[features_Onehot_encoded.columns] = features_Onehot_encoded
train_df= train_df.drop(train_df[categorical_feature] ,axis =1)
# train_df = train_df.dropna()
  




In [24]:
print(test_df.shape)
print(train_df.shape)

(1459, 43)
(1460, 207)


In [25]:
Onehot_Encoding = OneHotEncoder(sparse=False)
features_Onehot_encoded = pd.DataFrame(Onehot_Encoding.fit_transform(test_df[categorical_feature].astype(str)))
features_Onehot_encoded.columns = Onehot_Encoding.get_feature_names(categorical_feature)
test_df[features_Onehot_encoded.columns] = features_Onehot_encoded
test_df= test_df.drop(test_df[categorical_feature] ,axis =1)
# test_df = test_df.dropna()


In [26]:
train_df = train_df.fillna(0)
test_df = train_df.fillna(0)

In [27]:

ordinal_encoder = OrdinalEncoder()
ord_encoded_feature = pd.DataFrame(ordinal_encoder.fit_transform(train_df[ordinal_feature]))
train_df[ord_encoded_feature.columns] = ord_encoded_feature
train_df = train_df.drop(train_df[ordinal_feature] ,axis =1)

 

In [28]:
ordinal_encoder = OrdinalEncoder()
ord_encoded_feature = pd.DataFrame(ordinal_encoder.fit_transform(test_df[ordinal_feature]))
test_df[ord_encoded_feature.columns] = ord_encoded_feature
test_df = test_df.drop(test_df[ordinal_feature] ,axis =1)

# **Now we have to scale the Data**

In [29]:

scaler= StandardScaler()

train_to_scale = train_df.loc[:, train_df.columns != "SalePrice"]

In [30]:
scaled_df_array=scaler.fit_transform(train_to_scale)
scaled_train_df = pd.DataFrame(scaled_df_array, index=train_to_scale.index, columns=train_to_scale.columns)
scaled_train_df['SalePrice']=train_df['SalePrice']

In [31]:
scaled_df_array=scaler.fit_transform(test_df)
scaled_test_df = pd.DataFrame(scaled_df_array, index=test_df.index, columns=test_df.columns)

In [32]:

scaled_train_df.dropna(inplace=True)
scaled_test_df.dropna(inplace=True)

In [33]:
scaled_train_df['SalePrice']

Id
1       208500
2       181500
3       223500
4       140000
5       250000
         ...  
1455    185000
1456    175000
1457    210000
1458    266500
1459    142125
Name: SalePrice, Length: 1459, dtype: int64

In [34]:
scaled_train_df.dropna(axis=0, inplace=True)
scaled_test_df.dropna(axis=0, inplace=True)
X=scaled_train_df.drop('SalePrice' , axis=1)
y=scaled_train_df['SalePrice']


# **Now we use GridSerach CV to choose best parameteres for random forest and GradienBoosting model**

In [35]:
warnings.filterwarnings("ignore")
reg = GradientBoostingRegressor()
parameters = {'learning_rate': [0.01,0.02,0.03],
                  'subsample'    : [0.9, 0.5, 0.2, 0.1],
                  'n_estimators' : [100,500,1000],
                  'max_depth'    : [4,6,8]
                 }
grid_GBR = GridSearchCV(estimator=reg, param_grid = parameters, cv = 5, n_jobs=-1)
grid_GBR.fit(X, y)



/opt/conda/lib/python3.7/site-packages/sklearn/utils/validation.py:1692: FutureWarning: Feature names only support names that are all strings. Got feature names with dtypes: ['int', 'str']. An error will be raised in 1.2.
  FutureWarning,
/opt/conda/lib/python3.7/site-packages/sklearn/utils/validation.py:1692: FutureWarning: Feature names only support names that are all strings. Got feature names with dtypes: ['int', 'str']. An error will be raised in 1.2.
  FutureWarning,
/opt/conda/lib/python3.7/site-packages/sklearn/utils/validation.py:1692: FutureWarning: Feature names only support names that are all strings. Got feature names with dtypes: ['int', 'str']. An error will be raised in 1.2.
  FutureWarning,
/opt/conda/lib/python3.7/site-packages/sklearn/utils/validation.py:1692: FutureWarning: Feature names only support names that are all strings. Got feature names with dtypes: ['int', 'str']. An error will be raised in 1.2.
  FutureWarning,
/opt/conda/lib/python3.7/site-packages/sklea

GridSearchCV(cv=5, estimator=GradientBoostingRegressor(), n_jobs=-1,
             param_grid={'learning_rate': [0.01, 0.02, 0.03],
                         'max_depth': [4, 6, 8],
                         'n_estimators': [100, 500, 1000],
                         'subsample': [0.9, 0.5, 0.2, 0.1]})

In [36]:
print(" Results from Grid Search " )
print("\n The best estimator across ALL searched params:\n",grid_GBR.best_estimator_)
print("\n The best score across ALL searched params:\n",grid_GBR.best_score_)
print("\n The best parameters across ALL searched params:\n",grid_GBR.best_params_)

 Results from Grid Search 

 The best estimator across ALL searched params:
 GradientBoostingRegressor(learning_rate=0.01, max_depth=6, n_estimators=1000,
                          subsample=0.9)

 The best score across ALL searched params:
 0.8648471138517733

 The best parameters across ALL searched params:
 {'learning_rate': 0.01, 'max_depth': 6, 'n_estimators': 1000, 'subsample': 0.9}


In [37]:
from sklearn.ensemble import RandomForestRegressor


In [38]:
rnf = RandomForestRegressor()
parameters_rnf = {    'n_estimators' : [100,500,1000],
                       'max_depth'    : [4,6,8]
                 }
grid_rnf = GridSearchCV(estimator=rnf, param_grid = parameters_rnf, cv = 5, n_jobs=-1)
grid_rnf.fit(X, y)
# print("Accuracy: %.2f%% (%.2f%%)" % (results.mean()*100, results.std()*100))

/opt/conda/lib/python3.7/site-packages/sklearn/utils/validation.py:1692: FutureWarning: Feature names only support names that are all strings. Got feature names with dtypes: ['int', 'str']. An error will be raised in 1.2.
  FutureWarning,
/opt/conda/lib/python3.7/site-packages/sklearn/utils/validation.py:1692: FutureWarning: Feature names only support names that are all strings. Got feature names with dtypes: ['int', 'str']. An error will be raised in 1.2.
  FutureWarning,
/opt/conda/lib/python3.7/site-packages/sklearn/utils/validation.py:1692: FutureWarning: Feature names only support names that are all strings. Got feature names with dtypes: ['int', 'str']. An error will be raised in 1.2.
  FutureWarning,
/opt/conda/lib/python3.7/site-packages/sklearn/utils/validation.py:1692: FutureWarning: Feature names only support names that are all strings. Got feature names with dtypes: ['int', 'str']. An error will be raised in 1.2.
  FutureWarning,
/opt/conda/lib/python3.7/site-packages/sklea

GridSearchCV(cv=5, estimator=RandomForestRegressor(), n_jobs=-1,
             param_grid={'max_depth': [4, 6, 8],
                         'n_estimators': [100, 500, 1000]})

In [39]:
print(" Results from Grid Search " )
print("\n The best estimator across ALL searched params:\n",grid_rnf.best_estimator_)
print("\n The best score across ALL searched params:\n",grid_rnf.best_score_)
print("\n The best parameters across ALL searched params:\n",grid_rnf.best_params_)

 Results from Grid Search 

 The best estimator across ALL searched params:
 RandomForestRegressor(max_depth=8)

 The best score across ALL searched params:
 0.8385645136761329

 The best parameters across ALL searched params:
 {'max_depth': 8, 'n_estimators': 100}


In [40]:
test_df.dropna(inplace=True)
X_test=test_df[X.columns]

In [41]:
print(X.shape)
print(y.shape)

(1459, 206)
(1459,)


# Choose the best Parameters and use them to train and test

In [42]:
final_model=GradientBoostingRegressor(learning_rate=0.03, max_depth=6, n_estimators=500,
                          subsample=0.9)
final_model.fit(X,y)
y_pred=final_model.predict(X_test)


In [43]:
y_pred

array([576242.5775397 , 292935.99679066, 552647.51759859, 570002.40773689,
       560759.56013473, 570641.92510527, 282751.91295633, 558538.42189547,
       610386.63155271, 304328.15000325, 286361.1251827 , 565979.66974296,
       300916.17028617, 362143.33070611, 293140.48231054, 346978.41259804,
       292005.87281926, 363125.98974396, 301280.23376401, 288113.6332921 ,
       615905.02976373, 354321.15698729, 367098.2842276 , 292305.16224822,
       301250.29257979, 366699.784853  , 301828.53813616, 287634.85443569,
       289018.65310015, 354688.84622755, 581112.42760974, 352798.79058223,
       366320.69587233, 297262.78789666, 296806.99900633, 612034.68275221,
       353258.48480084, 296193.66806879, 300313.6995932 , 291698.6927407 ,
       298951.37287117, 292480.65876358, 292163.6931253 , 305597.92245802,
       299316.45761874, 285777.237821  , 573097.02302137, 302485.39855328,
       587233.34398583, 297799.16841554, 553168.45680102, 351243.96322251,
       288460.90523284, 3

In [44]:
submission = pd.DataFrame({'Id': X_test.index, 'SalePrice': y_pred})
submission.to_csv('submission.csv', index=False)
